In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/20020109tharindu/ML_Final_Project.git
%cd ML_Final_Project
!pip install imbalanced-learn -q

In [ ]:
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve
)

warnings.filterwarnings('ignore')

In [ ]:
# loading the preprocessed splits that team lead saved to drive
base_path = '/content/drive/MyDrive/FraudDetection/cleaned_data'

X_train = pd.read_csv(base_path + '/X_train.csv')
X_test  = pd.read_csv(base_path + '/X_test.csv')
y_train = pd.read_csv(base_path + '/y_train.csv').squeeze()
y_test  = pd.read_csv(base_path + '/y_test.csv').squeeze()

print('train size :', X_train.shape)
print('test size  :', X_test.shape)
print('fraud in train:', y_train.sum(), '| fraud in test:', y_test.sum())

In [ ]:
X_train.head()

In [ ]:
# quick look at class balance
print('Training set distribution (SMOTE balanced):')
print(y_train.value_counts())
print()
print('Test set distribution:')
print(y_test.value_counts())

In [ ]:
# train the logistic regression model
# using lbfgs solver because it handles L2 regularization well
# increasing max_iter to 1000 so it doesn't stop early
lr = LogisticRegression(solver='lbfgs', max_iter=1000, C=1.0, random_state=42)
lr.fit(X_train, y_train)

print('done training')

In [ ]:
# get predictions and probability scores on test set
y_pred = lr.predict(X_test)
y_prob = lr.predict_proba(X_test)[:, 1]  # probability of fraud class

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_prob)

print(f'Accuracy  : {acc:.4f}')
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')
print(f'F1-Score  : {f1:.4f}')
print(f'ROC-AUC   : {auc:.4f}')

In [ ]:
print(classification_report(y_test, y_pred, target_names=['Legit', 'Fraud']))

In [ ]:
# confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legit', 'Fraud'],
            yticklabels=['Legit', 'Fraud'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc:.4f}')
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Logistic Regression')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# precision recall curve — more useful than roc for imbalanced datasets
prec_vals, rec_vals, _ = precision_recall_curve(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(rec_vals, prec_vals, color='darkorange', lw=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.tight_layout()
plt.show()

In [ ]:
# feature coefficients — positive means pushes toward fraud, negative toward legit
coef = pd.Series(lr.coef_[0], index=X_train.columns)
top30 = pd.concat([coef.nsmallest(15), coef.nlargest(15)])

colors = ['steelblue' if v < 0 else 'crimson' for v in top30.values]

plt.figure(figsize=(9, 8))
top30.plot(kind='barh', color=colors)
plt.axvline(0, color='black', lw=0.8, linestyle='--')
plt.title('Feature Coefficients - Logistic Regression')
plt.xlabel('Coefficient')
plt.tight_layout()
plt.show()

In [ ]:
# try different thresholds and see which gives the best f1
# default is 0.5 but we can do better for fraud detection
thresh_range = np.arange(0.1, 0.9, 0.05)
rows = []

for t in thresh_range:
    preds = (y_prob >= t).astype(int)
    rows.append({
        'threshold' : round(t, 2),
        'precision' : precision_score(y_test, preds, zero_division=0),
        'recall'    : recall_score(y_test, preds, zero_division=0),
        'f1'        : f1_score(y_test, preds, zero_division=0)
    })

tdf    = pd.DataFrame(rows)
best_t = tdf.loc[tdf['f1'].idxmax(), 'threshold']
best   = tdf.loc[tdf['f1'].idxmax()]

print(f'best threshold : {best_t}')
print(f'  precision    : {best["precision"]:.4f}')
print(f'  recall       : {best["recall"]:.4f}')
print(f'  f1           : {best["f1"]:.4f}')

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(tdf['threshold'], tdf['precision'], label='Precision', color='steelblue')
plt.plot(tdf['threshold'], tdf['recall'],    label='Recall',    color='darkorange')
plt.plot(tdf['threshold'], tdf['f1'],        label='F1',        color='green')
plt.axvline(best_t, color='red', linestyle='--', label=f'best = {best_t}')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Threshold Tuning')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# final results using the best threshold
y_pred_final = (y_prob >= best_t).astype(int)

print('=== Final Results — Logistic Regression ===')
print(f'Threshold : {best_t}')
print(f'Accuracy  : {accuracy_score(y_test, y_pred_final):.4f}')
print(f'Precision : {precision_score(y_test, y_pred_final):.4f}')
print(f'Recall    : {recall_score(y_test, y_pred_final):.4f}')
print(f'F1-Score  : {f1_score(y_test, y_pred_final):.4f}')
print(f'ROC-AUC   : {auc:.4f}')